In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).
✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- Cell 22: V39 Ultimate (Ensemble: ResNet + EffNet + DeepLab) ---
# [FINAL BUILD - PLATINUM ENGINE]

import gc, os, csv, glob, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ---------------------------
# 0) Constants & Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"

# مسار الداتا سيت الجديدة التي تحتوي على الموديلات الثلاثة
MODELS_DIR = "/kaggle/input/ecg-final-models1"

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_NAMES)}
TARGET_H = 256
YOLO_INF_MAX = 1280       # pre-resize max side for YOLO inference
YOLO_CONF = 0.10          # lower conf to reduce missed detections
MAX_W = 2048              # clamp width for UNet input to avoid OOM
MAX_CACHE = 12            # patient cache size
DP_MAX_W = 1400           # Viterbi width threshold
DP_SMOOTH = 0.45

print(f"⚡ Device: {DEVICE}")

# ---------------------------
# 1) Read template + Dynamic lengths
# ---------------------------
if not os.path.exists(SAMPLE_PARQUET_PATH):
    raise FileNotFoundError("❌ sample_submission.parquet not found")

print("📦 Reading Parquet template ids...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
del template
gc.collect()

print(f"✅ Template rows: {len(template_ids):,}")

# Dynamic length map
patient_lengths = {}
print("🧠 Scanning template for patient lengths...")
for rid in tqdm(template_ids, desc="Scanning Lengths"):
    try:
        parts = rid.rsplit("_", 2)
        if len(parts) != 3: continue
        pid = str(parts[0]).strip()
        if pid.endswith(".0"): pid = pid[:-2]
        idx = int(parts[1])
        cur = patient_lengths.get(pid, 0)
        if idx + 1 > cur: patient_lengths[pid] = idx + 1
    except: pass
print(f"✅ Lengths mapped for {len(patient_lengths):,} patients.")

# ---------------------------
# 2) Index images + fs_map
# ---------------------------
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

fs_map = {}
if os.path.exists(TEST_CSV_PATH):
    try:
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        cols_low = {c.lower(): c for c in tdf.columns}
        id_c = next((cols_low[c] for c in cols_low if "id" in c), None)
        fs_c = next((cols_low[c] for c in cols_low if "fs" in c), None)
        if id_c and fs_c:
            fs_map = dict(zip(tdf[id_c].astype(str), tdf[fs_c].astype(str)))
        print(f"✅ fs_map loaded: {len(fs_map):,} items")
    except Exception as e:
        print(f"⚠️ fs_map read failed: {e}")

# ---------------------------
# 3) Load Models (Ensemble Mode)
# ---------------------------
print("⚙️ Loading models (Ensemble Mode)...")

# A. YOLO
yolo_model = None
CLASS_TO_LEAD_IDX = {}

def _normalize_lead_name(x: str) -> str:
    s = str(x).strip()
    s = s.replace("Lead", "").replace("lead", "").replace("_", "").replace("-", "").replace(" ", "")
    s = s.replace("AVR", "aVR").replace("AVL", "aVL").replace("AVF", "aVF")
    s = s.replace("AVr","aVR").replace("AVl","aVL").replace("AVf","aVF")
    return s

YOLO_PATH = f"{MODELS_DIR}/best.pt"
if os.path.exists(YOLO_PATH):
    try:
        yolo_model = YOLO(YOLO_PATH)
        print(f"✅ YOLO loaded: {YOLO_PATH}")
        
        # Build mapping
        if yolo_model.names:
            names = yolo_model.names
            items = list(names.items()) if isinstance(names, dict) else list(enumerate(names))
            for cid, cname in items:
                n = _normalize_lead_name(cname)
                if n in LEAD_TO_IDX:
                    CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[n]
            # Fallback fill
            for i in range(12): CLASS_TO_LEAD_IDX.setdefault(i, i)
    except Exception as e:
        print(f"⚠️ YOLO load failed: {e}")
else:
    print(f"⚠️ YOLO path not found: {YOLO_PATH}")

# B. Ensemble U-Nets
unet_models = []

try:
    # Model 1: ResNet50 (Original Phase 10)
    p1 = f"{MODELS_DIR}/best_model_phase10.pth"
    if os.path.exists(p1):
        m1 = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")
        m1.load_state_dict(torch.load(p1, map_location=DEVICE))
        unet_models.append(m1.to(DEVICE).eval())
        print("✅ Ensemble Model 1: ResNet50 Loaded.")
    else:
        print(f"❌ Missing: {p1}")

    # Model 2: EfficientNet-B3
    p2 = f"{MODELS_DIR}/best_model_efficientnet_ph10.pth"
    if os.path.exists(p2):
        m2 = smp.Unet(encoder_name="efficientnet-b3", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")
        m2.load_state_dict(torch.load(p2, map_location=DEVICE))
        unet_models.append(m2.to(DEVICE).eval())
        print("✅ Ensemble Model 2: EfficientNet-B3 Loaded.")
    else:
        print(f"❌ Missing: {p2}")

    # Model 3: DeepLabV3+
    p3 = f"{MODELS_DIR}/best_model_deeplab_ph10.pth"
    if os.path.exists(p3):
        m3 = smp.DeepLabV3Plus(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1)
        m3.load_state_dict(torch.load(p3, map_location=DEVICE))
        unet_models.append(m3.to(DEVICE).eval())
        print("✅ Ensemble Model 3: DeepLabV3+ Loaded.")
    else:
        print(f"❌ Missing: {p3}")

    if not unet_models:
        raise RuntimeError("❌ Fatal: No U-Net models loaded!")
        
    print(f"🔥 Ensemble Ready with {len(unet_models)} models.")

except Exception as e:
    print(f"❌ Model loading error: {e}")
    # Emergency fallback (empty list will trigger zeros later)
    unet_models = []

# ---------------------------
# 4) Helpers
# ---------------------------
def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path(pid):
    pid_s = clean_pid(pid)
    if pid_s in path_map: return path_map[pid_s]
    try: return path_map.get(str(int(float(pid_s))))
    except: return None

def safe_read_rgb(path):
    try:
        img = cv2.imread(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None
    except: return None

def preprocess_remove_grid_rgb(img_rgb):
    try:
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out = img_rgb.copy()
        out[mask > 0] = (255, 255, 255)
        return out
    except: return img_rgb

def apply_high_pass(data, cutoff=0.5, fs=500.0, order=3):
    try:
        if len(data) < order * 3: return data
        nyq = 0.5 * fs
        if nyq <= 0: return data
        wn = cutoff / nyq
        if not (0 < wn < 1): return data
        b, a = butter(order, wn, btype="high")
        return filtfilt(b, a, data)
    except: return data

def smart_einthoven_fix(leads):
    try:
        if 'I' in leads and 'II' in leads and 'III' in leads:
            l = min(len(leads['I']), len(leads['II']), len(leads['III']))
            I, II, III = leads['I'][:l], leads['II'][:l], leads['III'][:l]
            residual = (I + III) - II
            leads['II'][:l] += residual / 3.0
            leads['I'][:l]  -= residual / 3.0
            leads['III'][:l]-= residual / 3.0
    except: pass
    return leads

def estimate_grid_spacing_px(img_rgb):
    try:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        if gray.size == 0 or np.std(gray) < 3: return None
        ex = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        ey = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sx = np.sum(np.abs(ex), axis=0)
        sy = np.sum(np.abs(ey), axis=1)
        px, _ = find_peaks(sx, distance=10, prominence=np.percentile(sx, 75))
        py, _ = find_peaks(sy, distance=10, prominence=np.percentile(sy, 75))
        diffs = []
        if len(px) > 5:
            dx = np.diff(px)
            diffs.extend(dx[(dx > 8) & (dx < 120)])
        if len(py) > 5:
            dy = np.diff(py)
            diffs.extend(dy[(dy > 8) & (dy < 120)])
        if len(diffs) < 6: return None
        return float(np.median(np.array(diffs)))
    except: return None

def choose_ppmv(local_grid_px, resize_scale, raw_trace_px):
    if local_grid_px is None or not np.isfinite(local_grid_px) or local_grid_px < 5:
        return 250.0
    ppmv_A_orig = local_grid_px * 10.0
    ppmv_B_orig = local_grid_px * 2.0
    ppmv_A = ppmv_A_orig * resize_scale
    ppmv_B = ppmv_B_orig * resize_scale

    def score_candidate(ppmv):
        if ppmv <= 0: return 1e9
        sig = (raw_trace_px - np.median(raw_trace_px)) / ppmv
        sig = np.nan_to_num(sig)
        p2p = float(np.percentile(sig, 99) - np.percentile(sig, 1))
        if p2p <= 0: return 1e9
        if p2p < 0.2: return 1000.0 + (0.2 - p2p) * 1000.0
        if p2p > 8.0: return 1000.0 + (p2p - 8.0) * 300.0
        return abs(p2p - 2.5)

    sA, sB = score_candidate(ppmv_A), score_candidate(ppmv_B)
    ppmv = ppmv_A if sA <= sB else ppmv_B
    if not np.isfinite(ppmv) or ppmv < 10: return 250.0
    return float(ppmv)

# ---------------------------
# 5) Viterbi & YOLO Logic
# ---------------------------
def viterbi_dp(prob_map, smooth=0.45):
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6).astype(np.float32)
    dp = np.zeros_like(cost, dtype=np.float32)
    parent = np.zeros((H, W), dtype=np.int16)
    dp[:, 0] = cost[:, 0]
    parent[:, 0] = np.arange(H, dtype=np.int16)
    for t in range(1, W):
        prev = dp[:, t-1]
        c_m1 = np.roll(prev, 1);  c_m1[0] = 1e9
        c_0  = prev
        c_p1 = np.roll(prev, -1); c_p1[-1] = 1e9
        stacked = np.stack([c_m1 + smooth, c_0, c_p1 + smooth], axis=0)
        which = np.argmin(stacked, axis=0).astype(np.int16)
        best_prev = stacked[which, np.arange(H)]
        dp[:, t] = cost[:, t] + best_prev
        parent[:, t] = np.clip(np.arange(H, dtype=np.int16) + (which - 1), 0, H-1)
    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:, -1]))
    for t in range(W-2, -1, -1):
        path[t] = int(parent[path[t+1], t+1])
    return (H - path).astype(np.float32)

def viterbi_adaptive(prob_map):
    H, W = prob_map.shape
    if W <= 20 or H <= 5:
        path = np.argmax(prob_map, axis=0)
        return (H - path).astype(np.float32)
    if W <= DP_MAX_W:
        return viterbi_dp(prob_map, smooth=DP_SMOOTH)
    path = np.argmax(prob_map, axis=0).astype(np.float32)
    win = 21 if W >= 21 else (W if W % 2 == 1 else W-1)
    if win >= 5: path = savgol_filter(path, win, 2)
    return (H - path).astype(np.float32)

def get_crops_yolo_mapped(img_rgb):
    crops = [None] * 12
    if yolo_model is None: return crops
    h0, w0 = img_rgb.shape[:2]
    if h0 < 20 or w0 < 20: return crops
    scale = YOLO_INF_MAX / float(max(h0, w0))
    if scale < 1.0:
        w1, h1 = max(32, int(w0 * scale)), max(32, int(h0 * scale))
        img_inf = cv2.resize(img_rgb, (w1, h1))
    else:
        img_inf = img_rgb
        scale = 1.0
    best = {}
    try:
        res = yolo_model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
        if res and res[0].boxes is not None:
            boxes = res[0].boxes.data.detach().cpu().numpy()
            for b in boxes:
                x1, y1, x2, y2, conf, cls_id = b[:6]
                cls_id = int(cls_id)
                conf = float(conf)
                target_idx = CLASS_TO_LEAD_IDX.get(cls_id, cls_id)
                if not (0 <= target_idx < 12): continue
                x1o, y1o, x2o, y2o = x1 / scale, y1 / scale, x2 / scale, y2 / scale
                x1o, y1o = max(0, int(x1o)), max(0, int(y1o))
                x2o, y2o = min(w0, int(x2o)), min(h0, int(y2o))
                if x2o <= x1o + 10 or y2o <= y1o + 10: continue
                prev = best.get(target_idx)
                if (prev is None) or (conf > prev[0]):
                    best[target_idx] = (conf, (x1o, y1o, x2o, y2o))
        for li, (conf, (x1o, y1o, x2o, y2o)) in best.items():
            crops[li] = img_rgb[y1o:y2o, x1o:x2o]
        return crops
    except: return crops

# ---------------------------
# 6) Ensemble Inference Functions
# ---------------------------
def prepare_unet_batch(crops_rgb):
    tens_list, scales, widths = [], [], []
    for c in crops_rgb:
        h, w = c.shape[:2]
        if h < 5 or w < 5: continue
        sc = TARGET_H / float(h)
        nw = int(max(1, w * sc))
        if nw > MAX_W:
            nw = MAX_W
            sc = nw / float(w)
        rz = cv2.resize(c, (nw, TARGET_H))
        t = torch.from_numpy(rz).permute(2, 0, 1).float() / 255.0
        tens_list.append(t)
        scales.append(sc)
        widths.append(nw)
    if not tens_list: return None, None, None
    max_w = int(np.ceil(max(widths) / 32.0) * 32)
    max_w = min(max_w, MAX_W)
    batch = torch.zeros((len(tens_list), 3, TARGET_H, max_w), dtype=torch.float32, device=DEVICE)
    for i, t in enumerate(tens_list):
        w_i = min(t.shape[2], max_w)
        batch[i, :, :, :w_i] = t[:, :, :w_i].to(DEVICE)
    return batch, scales, widths

def predict_ensemble_probmaps(crops_rgb, models_list):
    """
    Predicts using MULTIPLE models + TTA and averages the probability maps.
    """
    batch, scales, widths = prepare_unet_batch(crops_rgb)
    if batch is None: return [], []

    accumulated_preds = None

    try:
        with torch.no_grad():
            for model in models_list:
                # Use autocast for speed
                with torch.amp.autocast('cuda', enabled=(DEVICE=="cuda")):
                    p1 = torch.sigmoid(model(batch))
                    p2 = torch.sigmoid(model(torch.flip(batch, [3]))) # Flip TTA
                
                # Combine TTA for this model
                preds_model = (p1 + torch.flip(p2, [3])) / 2.0
                
                if accumulated_preds is None:
                    accumulated_preds = preds_model
                else:
                    accumulated_preds += preds_model
        
        # Average across all models
        final_preds = accumulated_preds / len(models_list)
        final_preds = final_preds.detach().float().cpu().numpy()

        prob_maps = []
        for i in range(final_preds.shape[0]):
            w_i = widths[i]
            p_map = final_preds[i, 0, :, :w_i].astype(np.float32)
            
            # Artifact suppression
            col_means = np.mean(p_map, axis=0)
            global_median = np.median(col_means)
            mask = np.ones_like(col_means)
            mask[col_means > (global_median * 3.0 + 0.1)] = 0.1
            p_map = p_map * mask[None, :]
            
            prob_maps.append(p_map)

        return prob_maps, scales

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return [], []
        raise e

# ---------------------------
# 7) Patient Compute
# ---------------------------
patient_cache = OrderedDict()
def compute_patient_leads(pid, target_len):
    out = np.zeros((12, target_len), dtype=np.float32)
    path = get_image_path(pid)
    if not path: return out
    img = safe_read_rgb(path)
    if img is None: return out
    
    fs = float(fs_map.get(clean_pid(pid), 500.0))
    crops = get_crops_yolo_mapped(img)
    detected = [(i, c) for i, c in enumerate(crops) if c is not None and c.size > 200]
    if not detected: return out
    
    lead_indices = [i for i, _ in detected]
    crops_rgb = [preprocess_remove_grid_rgb(c) for _, c in detected]

    # --- ENSEMBLE PREDICTION ---
    prob_maps, scales = predict_ensemble_probmaps(crops_rgb, unet_models)
    
    if not prob_maps: return out

    global_grid = estimate_grid_spacing_px(img)
    leads_dict = {}
    
    for j in range(len(prob_maps)):
        lead_idx = lead_indices[j]
        prob = prob_maps[j]
        sc = scales[j]
        raw_px = viterbi_adaptive(prob)
        local_grid = estimate_grid_spacing_px(detected[j][1])
        if local_grid is None: local_grid = global_grid
        
        ppmv = choose_ppmv(local_grid, sc, raw_px)
        sig_mv = (raw_px - np.median(raw_px)) / ppmv
        sig_mv = np.nan_to_num(sig_mv, nan=0.0, posinf=0.0, neginf=0.0)
        
        if len(sig_mv) >= 11: sig_mv = savgol_filter(sig_mv, 11, 3)
        if len(sig_mv) >= 30: sig_mv = apply_high_pass(sig_mv, cutoff=0.5, fs=fs, order=3)
        
        sig_mv = np.clip(sig_mv, -7.0, 7.0)
        if len(sig_mv) > 0:
            leads_dict[LEAD_NAMES[lead_idx]] = resample(sig_mv, target_len).astype(np.float32)

    leads_dict = smart_einthoven_fix(leads_dict)
    for i, name in enumerate(LEAD_NAMES):
        if name in leads_dict: out[i] = np.nan_to_num(leads_dict[name], nan=0.0)
    return out

# ---------------------------
# 8) Streaming Write
# ---------------------------
print("🚀 Writing submission.csv (Platinum Ensemble)...")
with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing Rows"):
        try:
            parts = rid.rsplit("_", 2)
            if len(parts) != 3:
                writer.writerow([rid, "0.0000"])
                continue
            pid_raw, idx_raw, lead_name = parts
            pid = clean_pid(pid_raw)
            idx = int(idx_raw)
            
            t_len = patient_lengths.get(pid, 5000)
            if t_len <= 0 or t_len > 10000: t_len = 5000

            if pid in patient_cache:
                mtx = patient_cache[pid]
                patient_cache.move_to_end(pid)
            else:
                mtx = compute_patient_leads(pid, t_len)
                patient_cache[pid] = mtx
                if len(patient_cache) > MAX_CACHE:
                    patient_cache.popitem(last=False)
            
            lead_idx = LEAD_TO_IDX.get(lead_name, 0)
            val = 0.0
            if 0 <= idx < mtx.shape[1]:
                v = float(mtx[lead_idx][idx])
                if np.isfinite(v): val = v
            writer.writerow([rid, f"{val:.4f}"])
        except:
            writer.writerow([rid, "0.0000"])

del patient_cache
gc.collect()
torch.cuda.empty_cache()
print("✅ Done. submission.csv ready.")

⚡ Device: cuda
📦 Reading Parquet template ids...
✅ Template rows: 75,000
🧠 Scanning template for patient lengths...


Scanning Lengths: 100%|██████████| 75000/75000 [00:00<00:00, 916956.12it/s]

✅ Lengths mapped for 2 patients.
🗂️ Indexing images...


✅ Indexed images: 8,795
✅ fs_map loaded: 2 items
⚙️ Loading models (Ensemble Mode)...
✅ YOLO loaded: /kaggle/input/ecg-final-models1/best.pt
✅ Ensemble Model 1: ResNet50 Loaded.
✅ Ensemble Model 2: EfficientNet-B3 Loaded.
✅ Ensemble Model 3: DeepLabV3+ Loaded.
🔥 Ensemble Ready with 3 models.
🚀 Writing submission.csv (Platinum Ensemble)...


Processing Rows: 100%|██████████| 75000/75000 [00:04<00:00, 17875.47it/s]


✅ Done. submission.csv ready.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.94 MB
🎉 Ready to Submit.
